# CatBoost

https://catboost.ai/

CatBoost는 Yandex에서 개발한 Gradient Boosting 기반 알고리즘으로, 범주형 데이터 처리에 강점을 가진 머신러닝 모델이다.

이미 boosting의 기본 철학과, 실무 라이브러리가 그 철학을 어떻게 빠르고 강하게 구현하는지 보았다.
이제 CatBoost에서는 같은 boosting 계열 안에서도 범주형 데이터를 얼마나 자연스럽게 다룰 수 있는가라는 문제를 본다.

1. 좋은 boosting 모델은 성능만이 아니라 데이터 형태에 얼마나 잘 맞는지도 중요하다.
2. CatBoost는 범주형 데이터를 다룰 때 전처리 부담을 줄이도록 설계된 boosting 라이브러리이다.
3. 따라서 점수만 보는 것이 아니라, 범주형 열 지정 방식, Pool 객체, 특성 중요도까지 함께 봐야 CatBoost의 강점이 보인다.

이름 유래:
"CatBoost" = "Categorical + Boosting"

1. 범주형 데이터 처리에 강점
   * 원-핫 인코딩 없이도 범주형 데이터를 효과적으로 다룰 수 있다.
   * 범주형 값을 처리할 때 순서를 활용한 통계 기반 인코딩 방식을 사용하여, 타깃 누수를 줄이도록 설계되었다.
   * 이때 현재 샘플의 정답을 미리 사용하지 않도록 순서를 고려하여 인코딩함으로써 타깃 누수를 줄이도록 설계되었다.

2. 안정적인 학습 구조
   * 데이터를 보는 순서에 따라 인코딩 결과가 과하게 치우치지 않도록 permutation 기반 아이디어를 활용한다.
   * 이를 통해 데이터 순서에 대한 민감도를 줄이고, 보다 안정적으로 학습할 수 있다.
   * 예를 들어 같은 범주형 값이라도 현재 행의 정답까지 미리 반영해버리면 누수가 생길 수 있는데, CatBoost는 이런 문제를 줄이도록 설계되어 있다.

3. 빠른 학습 및 예측
   * 대규모 데이터셋에서도 효율적으로 동작하도록 설계되었다.
   * 특히 범주형 데이터 전처리를 따로 많이 하지 않아도 되어, 실무에서 전체 파이프라인을 단순하게 구성하기 좋다.

## 환경설정

In [1]:
%conda install catboost

Retrieving notices: done
Channels:
 - conda-forge
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\Playdata\AppData\Local\miniforge3\envs\ai_basic_env

  added / updated specs:
    - catboost


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    catboost-1.2.7             |  py312h5eb20be_1        48.6 MB  conda-forge
    fribidi-1.0.16             |       hfd05255_0          63 KB  conda-forge
    getopt-win32-0.1           |       h6a83c73_3          26 KB  conda-forge
    graphviz-14.1.2            |       h4c50273_0         1.2 MB  conda-forge
    gts-0.7.6                  |       h6b5321d_4         184 KB  conda-forge
    libgd-2.3.3                |      h4974f7c_12         163 KB  conda-forge
    narwhals-2.18.1            |     pyhcf101f3_1         274 KB  conda-forge
    numpy-1.26.4               |  py312h8753938_0         6.2 MB  co

C:\Users\Playdata\AppData\Local\miniforge3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'conda.anaconda.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


==> WARNING: A newer version of conda exists. <==
    current version: 26.1.0
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda




## 간단예제 구현

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = {
    'gender': ['Male', 'Female', 'Female', 'Male', 'Female'],
    'region': ['North', 'South', 'East', 'West', 'North'],
    'membership_type': ['Basic', 'Premium', 'Basic', 'Basic', 'Premium'],
    'age': [23, 35, 45, 50, 27],
    'purchased': [0, 1, 0, 0, 1]
}

df = pd.DataFrame(data)
df

,gender,region,membership_type,age,purchased
0,Male,North,Basic,23,0
1,Female,South,Premium,35,1
2,Female,East,Basic,45,0
3,Male,West,Basic,50,0
4,Female,North,Premium,27,1


In [ ]:
# 데이터 전처리
from sklearn.model_selection import train_test_split

X = df.drop('purchased', axis=1)
y = df['purchased']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
from catboost import Pool, CatBoostClassifier
from sklearn.metrics import accuracy_score

# 범주형 데이터 정의
cat_features = ['gender','region','membership_type']

# pool : CatBoost 전용 데이터 객체로 입력 데이터, 타깃, 범주형 열 정보 등을 함께 묶어 전달한다.
train_pool = Pool(X_train,y_train,cat_features=cat_features)
test_pool = Pool(X_test,y_test,cat_features=cat_features)

# 모델 학습
cat_clf = CatBoostClassifier(
    iterations=100,         # boosting round 수
    depth=3,                # 트리 깊이
    learning_rate=0.1,      # 학습률
    verbose=0
)

# Pool 객체를 전달하여 학습
cat_clf.fit(train_pool)

print('accuracy: ',accuracy_score(y_test,cat_clf.predict(test_pool)))

accuracy:  1.0


In [6]:
# 특성 중요도
simple_importance_df = pd.DataFrame({
    'feature' : X_train.columns,
    'importance' : cat_clf.get_feature_importance(train_pool),
}).sort_values('importance',ascending=False)

simple_importance_df

,feature,importance
2,membership_type,83.249902
3,age,7.961470
0,gender,7.901964
1,region,0.886663


In [7]:
# 각 클래스에 속할 확률
print(cat_clf.predict_proba(test_pool))

[[0.28663273 0.71336727]]


## Adult Income

In [8]:
# 데이터로드
data_df = pd.read_csv('data/adult_income.csv')
data_df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [15]:
# 데이터 분할
X = data_df.drop('income', axis=1)
y = data_df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [16]:
# 일반 모델을 사용한다면 범주형에 대한 전처리 필요 (logisticr regression)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

categorical_features = ['workclass','education','marital-status','occupation',
                        'relationship','race','sex','native-country']
numeric_features = [col for col in X_train.columns if col not in categorical_features]

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'),categorical_features),
    ('num','passthrough', numeric_features)
])

bassline_model = Pipeline([
    ('preprocessor',preprocessor),
    ('model', LogisticRegression(max_iter=5000))
])

bassline_model.fit(X_train,y_train)

c:\Users\Playdata\AppData\Local\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [18]:
# 모델 학습
categorical_features = ['workclass','education','marital-status','occupation',
                        'relationship','race','sex','native-country']

train_pool = Pool(X_train,y_train,cat_features=categorical_features)
test_pool = Pool(X_test,y_test,cat_features=categorical_features)

cat_clf = CatBoostClassifier(iterations=100,learning_rate=0.1,depth=8,verbose=50)
cat_clf.fit(train_pool)

0:	learn: 0.6418488	total: 38ms	remaining: 3.76s
50:	learn: 0.2967996	total: 2.02s	remaining: 1.94s
99:	learn: 0.2807865	total: 3.87s	remaining: 0us


In [19]:
print('accuracy_score:',accuracy_score(y_test,cat_clf.predict(X_test)))

accuracy_score: 0.8730231844004299


In [20]:
# 특성 중요도
income_importance_df = pd.DataFrame({
    'feature' : X_train.columns,
    'importance' : cat_clf.get_feature_importance(train_pool),
}).sort_values('importance',ascending=False)

income_importance_df

,feature,importance
7,relationship,24.850726
10,capital-gain,14.559165
4,education-num,11.872360
0,age,11.816145
6,occupation,8.769089
12,hours-per-week,7.744278
11,capital-loss,5.899741
5,marital-status,4.118729
1,workclass,2.840465
3,education,2.380873
